# 实验 G 修复版：跳过已完成组，恢复未完成训练

默认检查留出 1、2、3 的 real / zero / shuffle。完整组直接跳过；未训练完的组从 last.ckpt 恢复模型、EMA、优化器和训练进度，继续至原配置的总计 100 epochs；尚未开始的组使用原初始权重和配对缓存训练。训练已结束但生成文件不完整时，只补模型导出及生成。

当前下载结果预期为 7 组完整、第三折 zero 已保存 47 个完整 epochs（epoch=46），第三折 shuffle 尚未开始。运行时以服务器实际文件为准。

**先确认原 G 训练进程或旧 Notebook 内核已停止，再使用新内核运行本文件。SSH 断线不一定终止服务器任务，避免两个任务同时写同一目录。**

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from pathlib import Path
import sys

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deepspatial/core.py").is_file() or (p / "HM_DeepSpital/deepspatial/core.py").is_file())
if (PROJECT / "HM_DeepSpital").is_dir():
    PROJECT = PROJECT / "HM_DeepSpital"
sys.path.insert(0, str(PROJECT))
from examples.resume_g import inspect_results, resume_experiment


In [ ]:
SOURCE_ROOT = PROJECT / "outputs/s_holdouts_1_2_3"
OUTPUT_ROOT = PROJECT / "outputs/g_holdouts_1_2_3"  # 必须指向原输出目录
HOLDOUTS = [1, 2, 3]
MODES = ["real", "zero", "shuffle"]
DEVICE = "auto"


In [ ]:
status = inspect_results(OUTPUT_ROOT, HOLDOUTS, MODES)
display(status)


action 含义：`skip` 已完整，完全跳过；`resume_training` 从最后完整 epoch 继续；`train_new` 尚未开始；`reconstruct` 已满训练轮数，只修复导出和生成。

续跑使用原 shared/run_config.json，不改变学习率、模型结构、物理尺度、配对、S 抽样或原总轮数。会核对原代码、配对、来源 S 文件、条件和初始权重身份；不一致时报错，不自动换配置。续跑日志保存到新的 checkpoints/logs/version_*，原日志保留。

In [ ]:
final_status = resume_experiment(
    source_root=SOURCE_ROOT, output_root=OUTPUT_ROOT,
    holdouts=HOLDOUTS, modes=MODES, device=DEVICE,
)
display(final_status)
assert (final_status.action == "skip").all(), "仍有未完成组，请查看上方状态"


再次运行本文件会重新检查并跳过完整组。若再次中断，可在原进程停止后用新内核重复运行。生成文件采用临时文件写完后替换；续跑记录保存于各组 resume_history.jsonl。

从 epoch 边界恢复时，可能重算断线前未保存的部分 batch；随机数流不保证与从未中断的运行逐步相同。不要删除或替换原 shared 配对缓存、S 来源结果、last.ckpt，也不要将总 epochs 改为剩余轮数。